** Feature Engineering**

In [9]:
import pandas as pd

# Load data
df = pd.read_csv('data_encoded.csv')

# === Fitur Rasio (2) ===
# 1. Rasio jam kerja terhadap skor depresi (PHQ-9)
# Mengukur seberapa besar beban kerja dibandingkan tingkat gejala depresi
df['work_phq9_ratio'] = df['work_hours_per_week'] / df['phq9_score'].clip(lower=1)

# 2. Rasio jam kerja terhadap skor kecemasan (GAD-7)
# Mengukur korelasi beban kerja terhadap tingkat kecemasan
df['work_gad7_ratio'] = df['work_hours_per_week'] / df['gad7_score'].clip(lower=1)

# 3. Rasio PHQ-9 terhadap GAD-7
# Mengukur beban mental lebih dominan ke Depresi atau Kecemasan
df['depression_anxiety_ratio'] = df['phq9_score'] / df['gad7_score'].clip(lower=1)

# === Fitur Binning (2) ===
# 1. Binning Usia menjadi kelompok generasi
df['age_group'] = pd.cut(df['age'],
                         bins=[0, 12, 17, 59, 100],
                         labels=['Anak-anak', 'Remaja', 'Dewasa', 'Lansia'])

# 2. Binning Jam Kerja (Quartile) untuk kategori beban kerja
df['workload_category'] = pd.qcut(df['work_hours_per_week'], q=4,
                                 labels=['Light', 'Normal', 'Heavy', 'Extreme'])


# === Fitur Agregasi (2) ===
# 1. Total Mental Health Burden (Agregasi skor PHQ-9 dan GAD-7)
# Memberikan gambaran beban psikologis total dari responden
df['total_mental_burden'] = df['phq9_score'] + df['gad7_score']

# 2. Rata-rata Skor per Jam Kerja
# Melihat intensitas beban mental yang dirasakan per jam kerja yang dilakukan
df['mental_intensity_per_hour'] = df['total_mental_burden'] / df['work_hours_per_week'].clip(lower=1)

# 3. Physical Resilience Score
# Melihat pengaruh olahraga dan tidur yang cukup terhadap daya tahan burnout.
df['physical_resilience_score'] = df['sleep_hours_per_night'] + df['exercise_days_per_week']

# === Save file ===
df.to_csv('data_features.csv', index=False)

print("Fitur baru berhasil dibuat sesuai kolom yang tersedia!")
print(df[['work_phq9_ratio', 'total_mental_burden', 'mental_intensity_per_hour']].head())

Fitur baru berhasil dibuat sesuai kolom yang tersedia!
   work_phq9_ratio  total_mental_burden  mental_intensity_per_hour
0         2.894737                   31                   0.563636
1         5.500000                   14                   0.318182
2         5.625000                   17                   0.377778
3         3.857143                   25                   0.462963
4         6.125000                    8                   0.163265


**Korelasi Antar Fitur Baru**

In [8]:
# Daftar semua fitur baru (8 fitur)
fitur_baru = [
    'work_phq9_ratio',
    'work_gad7_ratio',
    'depression_anxiety_ratio',
    'age_group',
    'workload_category',
    'total_mental_burden',
    'mental_intensity_per_hour',
    'physical_resilience_score'
]

print("Hasil Korelasi Fitur Baru terhadap Burnout Level:")
print("-" * 60)

for f in fitur_baru:
    if f in df.columns:
        # Cek jika tipe data kategorikal (untuk hasil binning)
        if df[f].dtype.name == 'category':
            # Ubah ke kode numerik sementara untuk hitung korelasi
            korelasi = df[f].cat.codes.corr(df["burnout_level_ord"])
        else:
            korelasi = df[f].corr(df["burnout_level_ord"])

        print(f"{f:30} : corr = {korelasi:.3f}")
    else:
        print(f"{f:30} : Kolom tidak ditemukan!")

Hasil Korelasi Fitur Baru terhadap Burnout Level:
------------------------------------------------------------
work_phq9_ratio                : corr = -0.570
work_gad7_ratio                : corr = -0.583
depression_anxiety_ratio       : corr = 0.018
age_group                      : corr = nan
workload_category              : corr = 0.420
total_mental_burden            : corr = 0.781
mental_intensity_per_hour      : corr = 0.743
physical_resilience_score      : corr = -0.319


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
